# Joshi Part 7: Exotic Engine - Path-Dependent Options

Based on **"The Concepts and Practice of Mathematical Finance"** by Mark S. Joshi.

Joshi's ExoticEngine extends the simple MC pricer to handle **path-dependent**
derivatives whose payoffs depend on the entire price trajectory, not just the
terminal value. This notebook covers:

1. **Asian options** - Payoff depends on the average price
2. **Barrier options** - Payoff depends on whether a barrier is breached
3. **Lookback options** - Payoff depends on the maximum or minimum price

We use RustQuant's GBM engine for fast path generation.

In [ ]:
import math
import statistics
from RustQuant.stochastics import GeometricBrownianMotion
from RustQuant.instruments import BlackScholesMerton, OptionType

spot = 100.0
strike = 100.0
rate = 0.05
vol = 0.20
T = 1.0
n_paths = 100_000
n_steps = 252  # Daily monitoring

# Generate paths once (all exotics use the same paths for fair comparison)
gbm = GeometricBrownianMotion(mu=rate, sigma=vol)
traj = gbm.simulate(x0=spot, t_end=T, n_steps=n_steps, n_paths=n_paths, parallel=True)
paths = traj.paths
df = math.exp(-rate * T)

print(f"Generated {len(paths)} paths with {len(paths[0])} steps each")

## Reference: Vanilla European

For comparison, we price a vanilla European call using both MC and analytics.

In [ ]:
vanilla_call_mc = df * sum(max(p[-1] - strike, 0) for p in paths) / len(paths)
vanilla_put_mc = df * sum(max(strike - p[-1], 0) for p in paths) / len(paths)

bsm_call = BlackScholesMerton(
    underlying_price=spot, strike_price=strike, volatility=vol,
    risk_free_rate=rate, cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)

print(f"Vanilla Call - MC: {vanilla_call_mc:.4f}, BS: {bsm_call.price():.4f}")
print(f"Vanilla Put  - MC: {vanilla_put_mc:.4f}")

## 1. Asian Options

The payoff depends on the **average** price over the option's life:

$$\text{Asian Call} = \max\left(\bar{S} - K, 0\right)$$

where $\bar{S}$ is the arithmetic or geometric average.

**Key insight (Joshi):** Averaging reduces effective volatility, so Asian options
are always cheaper than their vanilla counterparts.

In [ ]:
# Arithmetic average Asian
asian_arith_call = df * sum(
    max(statistics.mean(p) - strike, 0) for p in paths
) / len(paths)

asian_arith_put = df * sum(
    max(strike - statistics.mean(p), 0) for p in paths
) / len(paths)

# Geometric average Asian
def geo_mean(values):
    log_sum = sum(math.log(v) for v in values if v > 0)
    return math.exp(log_sum / len(values))

asian_geo_call = df * sum(
    max(geo_mean(p) - strike, 0) for p in paths
) / len(paths)

# Floating strike Asian: payoff = max(S_T - S_avg, 0)
asian_float_call = df * sum(
    max(p[-1] - statistics.mean(p), 0) for p in paths
) / len(paths)

print("Asian Options (K=100):")
print(f"  Arithmetic Avg Call: {asian_arith_call:.4f}  (vanilla: {vanilla_call_mc:.4f})")
print(f"  Arithmetic Avg Put:  {asian_arith_put:.4f}  (vanilla: {vanilla_put_mc:.4f})")
print(f"  Geometric Avg Call:  {asian_geo_call:.4f}")
print(f"  Floating Strike Call: {asian_float_call:.4f}")
print(f"\n  -> Asian < Vanilla (averaging reduces volatility exposure)")

## 2. Barrier Options

The payoff depends on whether the price **crosses a barrier** during the option's life.

- **Knock-Out**: Becomes worthless if barrier is breached
- **Knock-In**: Only activates when barrier is breached

**Key relation (Joshi):** Knock-In + Knock-Out = Vanilla

In [ ]:
barrier_up = 120.0
barrier_down = 80.0

def barrier_payoff(path, strike, barrier, direction, knock_type):
    """Compute barrier option payoff."""
    s_t = path[-1]
    vanilla_payoff = max(s_t - strike, 0)  # Call payoff
    
    if direction == "up":
        breached = any(s >= barrier for s in path)
    else:  # down
        breached = any(s <= barrier for s in path)
    
    if knock_type == "out":
        return 0.0 if breached else vanilla_payoff
    else:  # in
        return vanilla_payoff if breached else 0.0

# Up barriers
uo_call = df * sum(barrier_payoff(p, strike, barrier_up, "up", "out") for p in paths) / len(paths)
ui_call = df * sum(barrier_payoff(p, strike, barrier_up, "up", "in") for p in paths) / len(paths)

# Down barriers
do_call = df * sum(barrier_payoff(p, strike, barrier_down, "down", "out") for p in paths) / len(paths)
di_call = df * sum(barrier_payoff(p, strike, barrier_down, "down", "in") for p in paths) / len(paths)

print(f"Barrier Options (Call, K={strike}):")
print(f"\n  Upper Barrier = {barrier_up}:")
print(f"    Up-and-Out:   {uo_call:.4f}")
print(f"    Up-and-In:    {ui_call:.4f}")
print(f"    Sum (≈ Vanilla): {uo_call + ui_call:.4f}  (vanilla: {vanilla_call_mc:.4f})")
print(f"\n  Lower Barrier = {barrier_down}:")
print(f"    Down-and-Out: {do_call:.4f}")
print(f"    Down-and-In:  {di_call:.4f}")
print(f"    Sum (≈ Vanilla): {do_call + di_call:.4f}  (vanilla: {vanilla_call_mc:.4f})")
print(f"\n  -> Knock-In + Knock-Out ≈ Vanilla (parity)")

## 3. Lookback Options

The payoff depends on the **extremum** of the price path:

- **Fixed strike call**: $\max(S_{\max} - K, 0)$
- **Floating strike call**: $S_T - S_{\min}$

Lookback options are the most expensive because they provide "perfect hindsight".

In [ ]:
# Fixed strike lookback
lookback_fixed_call = df * sum(
    max(max(p) - strike, 0) for p in paths
) / len(paths)

lookback_fixed_put = df * sum(
    max(strike - min(p), 0) for p in paths
) / len(paths)

# Floating strike lookback
lookback_float_call = df * sum(
    max(p[-1] - min(p), 0) for p in paths
) / len(paths)

lookback_float_put = df * sum(
    max(max(p) - p[-1], 0) for p in paths
) / len(paths)

print("Lookback Options (K=100):")
print(f"\n  Fixed Strike:")
print(f"    Call max(S_max - K, 0): {lookback_fixed_call:.4f}  (vanilla: {vanilla_call_mc:.4f})")
print(f"    Put  max(K - S_min, 0): {lookback_fixed_put:.4f}  (vanilla: {vanilla_put_mc:.4f})")
print(f"\n  Floating Strike:")
print(f"    Call (S_T - S_min):     {lookback_float_call:.4f}")
print(f"    Put  (S_max - S_T):     {lookback_float_put:.4f}")

## 4. Price Ordering

Joshi emphasizes the intuitive ordering of option prices:

In [ ]:
results = [
    ("Asian (arith avg)", asian_arith_call),
    ("Vanilla", vanilla_call_mc),
    ("Up-and-Out (B=120)", uo_call),
    ("Down-and-Out (B=80)", do_call),
    ("Lookback (fixed K)", lookback_fixed_call),
    ("Lookback (float K)", lookback_float_call),
]

results.sort(key=lambda x: x[1])

print("Price Ordering (Calls, K=100, cheapest to most expensive):")
print(f"{'Option Type':<25} {'Price':>10}")
print("-" * 35)
for name, price in results:
    print(f"{name:<25} {price:>10.4f}")

print("\nIntuition:")
print("  - Barriers reduce price (you can lose the option)")
print("  - Averaging reduces price (lower effective vol)")
print("  - Lookbacks increase price (optimal exercise hindsight)")

## Summary

| Joshi Concept | Implementation | Key Insight |
|---------------|---------------|-------------|
| ExoticEngine | RustQuant GBM + Python payoffs | Separate path generation from payoff |
| Asian options | Arithmetic/geometric mean of path | Averaging reduces vol → cheaper |
| Barrier options | Check if barrier breached on path | Knock-In + Knock-Out = Vanilla |
| Lookback options | Extract min/max from path | Perfect hindsight → most expensive |